# 04 · CoT(사고사슬) 전략 검증

이 노트북은 **CoT(Chain-of-Thought, 답을 내기 전에 단계별로 추론하게 하는 방식)** 를
언제 쓰고 언제 피하는 게 좋은지, 그리고 형식·모델 크기·창의성이라는 축에서 결과가 어떻게 달라지는지를
`gpt-5-nano`(+ 구형 `gpt-4.1-nano`)로 **실제로 돌려보며** 확인한다.

검증하는 팁 요약(1줄씩):

- **Tip 14 — 직관형 문제엔 CoT가 오히려 손해**: 바로 답이 보이는 문제에 억지로 단계별 설명을 붙이면 정답률이 떨어질 수 있다.
- **Tip 15 — CoT는 서식에 민감**: 줄바꿈·불릿·마침표 같은 *형식만* 바뀌어도 CoT의 최종 답이 달라질 수 있다.
- **Tip 21 — 작은 모델엔 CoT가 도움**: 작은 모델(`gpt-4.1-nano`)은 "단계별로 풀어라"라고 명시해야 정답률이 올라간다.
- **Tip 30 — 무관한 두 개념 융합**: 서로 상관없는 두 영역을 반드시 결합하라고 시키면, 평범한 요청으론 안 나오는 발상이 나온다.
- **Tip 39 — 가상 온도(Virtual Temperature)**: temperature를 못 바꾸는 `gpt-5-nano`에서, 한 프롬프트 안에서
  "발산 → 수렴"을 텍스트로 전환한다.

> ⚠️ 실행 주의: 아래 셀들은 **실제 OpenAI API를 호출**한다. 셀을 Run 하면 크레딧이 소모된다.
> 정답률 측정 셀은 같은 문제를 여러 번(N회 루프) 돌리므로 호출 수가 늘어난다 — N을 조절해서 쓰라.

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## Tip 14 — 직관형 문제엔 CoT가 오히려 손해

**요지**: 사람이 보자마자 답을 아는 직관형 문제(짧은 패턴의 다음 항, 단순한 공간 지각 등)에
"step-by-step으로 설명하라"를 붙이면, 모델이 스스로 만든 잘못된 추론 과정을 따라가다가
**정답률이 오히려 떨어질 수 있다**.

**무엇을 어떻게 검증하나**:
- 직관형 문제 하나(숫자 패턴의 다음 항)를 준비하고 정답을 미리 알아 둔다.
- (A) *"정답 숫자 하나만"* vs (B) *"단계별 추론을 모두 서술한 뒤 정답"* 두 프롬프트를 각각 **N회** 돌린다.
- `ask_meta` 로 매 호출의 정답 여부와 latency(응답 지연 시간)를 모아 **정답률**과 **평균 latency**를 비교한다.
- 관찰 포인트: B(긴 CoT)가 A보다 **정답률은 같거나 낮은데 latency는 확실히 크면**, "직관형엔 CoT가 손해"가 재현된 것.

In [ ]:
# Tip 14 실험 — 직관형 패턴 문제에서 "바로 답" vs "단계별 CoT" 정답률/latency 비교
import re

# 직관형 문제: 2,4,8,16,? (다음 항 = 32). 사람은 즉답하지만 CoT는 삼천포로 빠지기 쉽다.
PROBLEM = "Sequence: 2, 4, 8, 16, ?  What single number comes next?"
GOLD = "32"
N = 5  # 각 조건당 호출 횟수 (크레딧 절약하려면 줄여라)

# (A) 직관 모드: 오직 숫자 하나만
PROMPT_A = PROBLEM + "\nAnswer with ONLY the number, nothing else."
# (B) 강제 CoT 모드: 장황하게 단계별로 서술한 뒤 답
PROMPT_B = (
    PROBLEM
    + "\nThink step by step. Explain every step of your reasoning in detail first, "
    "then on the last line write 'Answer: <number>'."
)

def extract_last_number(text):
    """응답에서 마지막으로 등장한 정수를 정답 후보로 뽑는다."""
    nums = re.findall(r"-?\d+", text or "")
    return nums[-1] if nums else None

def run_trials(prompt, n):
    correct, latencies = 0, []
    for _ in range(n):
        m = ask_meta(prompt, max_completion_tokens=2000)  # CoT는 길 수 있어 넉넉히
        latencies.append(m["latency"])
        if extract_last_number(m["text"]) == GOLD:
            correct += 1
    return correct, sum(latencies) / len(latencies)

acc_a, lat_a = run_trials(PROMPT_A, N)
acc_b, lat_b = run_trials(PROMPT_B, N)

print(f"(A) 바로 답만   : 정답 {acc_a}/{N}  | 평균 latency {lat_a:.2f}s")
print(f"(B) 강제 CoT    : 정답 {acc_b}/{N}  | 평균 latency {lat_b:.2f}s")
print("\n관전 포인트: B가 A보다 정답률이 같거나 낮으면서 latency가 크면 → '직관형엔 CoT가 독' 재현")

## Tip 15 — CoT의 서식 민감성

**요지**: 내용은 그대로 두고 **형식만**(줄바꿈 유무, 불릿 유무, 마침표 유무) 바꿔도 CoT의 최종 답이 달라질 수 있다.
CoT는 프롬프트의 표면 형태에 예민하다.

**무엇을 어떻게 검증하나**:
- 연산 순서를 헷갈리게 만드는, 함정이 있는 워드 프라블럼(서술형 계산 문제) 하나를 고정한다.
- **의미는 동일**하고 서식만 다른 4개 변형(한 줄 / 줄바꿈 / 불릿 / 마침표 제거)을 만든다.
- 4개를 같은 모델에 돌려 최종 숫자 답을 뽑고, **답이 서로 갈리는지(변동성)** 를 표로 관찰한다.
- 관찰 포인트: 4개 답이 전부 같으면 안정적이고, **하나라도 다르면 서식 민감성이 드러난 것**.

In [ ]:
# Tip 15 실험 — 의미 동일 / 서식만 다른 CoT 프롬프트 4종의 답 변동성
import re

# 함정 워드 프라블럼: 정답은 (3+5)*2 = 16 이 아니라, 문장을 잘 읽어야 하는 순서 문제.
# "밥은 사과 3개로 시작해 5개를 더 받고, 그 총합의 2배를 세었다" -> (3+5)*2 = 16
CORE = ("Bob starts with 3 apples, then receives 5 more, "
        "and finally counts double the running total. How many apples did he count?")

variants = {
    "v1 한 줄":      f"{CORE} Let's think step by step and give the final number.",
    "v2 줄바꿈":     f"{CORE}\nLet's think step by step.\nThen give the final number.",
    "v3 불릿":       f"{CORE}\n- think step by step\n- show each arithmetic step\n- give the final number",
    "v4 마침표제거":  f"{CORE.rstrip('.')} lets think step by step then give the final number",
}

def last_num(t):
    n = re.findall(r"-?\d+", t or "")
    return n[-1] if n else "?"

print(f"{'변형':<14}{'최종답':<8}응답 앞부분")
print("-" * 80)
answers = {}
for name, p in variants.items():
    txt = ask(p, max_completion_tokens=2000)
    a = last_num(txt)
    answers[name] = a
    preview = " ".join((txt or "").split())[:48]
    print(f"{name:<14}{a:<8}{preview}")

uniq = set(answers.values())
print("\n서로 다른 최종답 개수:", len(uniq), "->", sorted(uniq))
print("관전 포인트: 개수가 2 이상이면 → 내용 동일한데 서식만으로 답이 갈린 '서식 민감성' 재현")

## Tip 21 — 작은 모델(SLM)엔 CoT가 도움

**요지**: 작고 오래된 모델은 속으로 추론하지 못한 채 바로 답을 내면 자주 틀린다.
**"단계별로 풀어서 써라"** 라고 명시적으로 시키면 정답률이 올라간다.
(Tip 14와 정반대 상황임에 주목 — 대상이 작은 모델이고, 문제도 직관형이 아니라 여러 단계를 거치는 계산이라 CoT가 도움이 된다.)

**무엇을 어떻게 검증하나**:
- 대상은 구형 소형 모델 `OLD_MODEL`(gpt-4.1-nano).
- 여러 자리 곱셈이나 다단계 워드 프라블럼처럼 **암산으로 풀기 어려운** 문제를 준비한다(정답 미리 확보).
- (A) *"답만"* vs (B) *"계산 과정을 단계별로 모두 쓴 뒤 답"* 을 각각 N회 돌려 정답률을 비교한다.
- 관찰 포인트: **B(CoT)가 A보다 정답률이 뚜렷이 높으면** "작은 모델엔 CoT가 도움"이 재현된 것.

In [ ]:
# Tip 21 실험 — 구형 소형 모델(gpt-4.1-nano): CoT 없음 vs 단계별 강제 정답률 비교
import re

# 암산이 힘든 다단계 문제: 23 * 17 + 89 = 391 + 89 = 480
PROBLEM = "Compute 23 * 17 + 89. Give the exact integer."
GOLD = "480"
N = 5

PROMPT_A = PROBLEM + " Reply with ONLY the final integer, no words."
PROMPT_B = (PROBLEM + " Solve it step by step: first the multiplication, then the addition. "
            "Show each step, then on the last line write 'Answer: <integer>'.")

def last_num(t):
    n = re.findall(r"-?\d+", t or "")
    return n[-1] if n else None

def acc(prompt, n):
    c = 0
    for _ in range(n):
        # 구형 모델은 gpt-5 전용 파라미터 없이 model=OLD_MODEL 로만 호출
        txt = ask(prompt, model=OLD_MODEL, max_completion_tokens=300)
        if last_num(txt) == GOLD:
            c += 1
    return c

a = acc(PROMPT_A, N)
b = acc(PROMPT_B, N)
print(f"모델: {OLD_MODEL}")
print(f"(A) 답만 강요     : 정답 {a}/{N}")
print(f"(B) 단계별 강제CoT : 정답 {b}/{N}")
print("\n관전 포인트: B가 A보다 정답률이 높으면 → 'SLM은 중얼거리게 시켜야 답을 낸다' 재현")

## Tip 30 — 무관한 두 개념 융합 (상상력 끌어내기)

**요지**: "좋은 아이디어 줘" 같은 평범한 요청은 누구나 떠올리는 뻔한 답을 준다.
서로 **완전히 무관한 두 영역을 반드시 결합**하라고 조건을 걸면,
평범한 요청으론 나오지 않는 이질적이고 독창적인 결과가 나온다.

**무엇을 어떻게 검증하나**:
- (A) 평범: *"스타트업 아이디어 3개 줘"*
- (B) 융합 강제: *"심해 생물학 + 세무회계, 이 무관한 두 개념을 반드시 결합한 스타트업 아이디어 3개"*
- 두 결과를 `compare` 로 나란히 출력하고, `keyword_hits` 로 B가 실제로 두 영역의 용어를 **모두** 끌어왔는지 채점한다.
- 관찰 포인트: B가 A보다 구체적이고 이질적이며, 두 영역의 키워드가 함께 등장하면 "무관한 개념 융합"이 작동한 것.

In [ ]:
# Tip 30 실험 — 평범한 아이디어 vs 이질적 두 개념 융합 강제
SYSTEM = "You are a startup idea generator. Be concrete: give a name + one-line pitch per idea."

# (A) 평범한 요청
prompt_a = "Give me 3 startup ideas."

# (B) 이질적 두 개념 융합 강제 (바다 끓이기)
prompt_b = (
    "Fuse two completely unrelated domains — DEEP-SEA BIOLOGY and TAX ACCOUNTING — "
    "into 3 startup ideas. Every idea MUST visibly draw from BOTH domains at once. "
    "Be weird and specific."
)

text_a = ask(prompt_a, system=SYSTEM, max_completion_tokens=1200)
text_b = ask(prompt_b, system=SYSTEM, max_completion_tokens=1200)

compare("(A) 평범한 요청", text_a, "(B) 심해생물학 + 세무회계 융합 강제", text_b)

# 두 영역 용어가 실제로 함께 등장했는지 채점 (융합이 말로만 그치지 않았는지)
score_a = keyword_hits(text_a, ["deep-sea", "biology", "tax", "accounting"])
score_b = keyword_hits(text_b, ["deep-sea", "biology", "tax", "accounting"])
print("A 두영역 키워드 적중:", score_a["score"], "/", score_a["total"], score_a["hits"])
print("B 두영역 키워드 적중:", score_b["score"], "/", score_b["total"], score_b["hits"])
print("\n관전 포인트: B의 적중이 A보다 높고, 아이디어가 이질적으로 구체적이면 → '상상력 강제' 작동")

## Tip 39 — 가상 온도(Virtual Temperature)

**요지**: `gpt-5-nano`는 **temperature를 바꿀 수 없다(1로 고정)**. 그래서 "발산/수렴"을 파라미터로 조절하지 못한다.
대신 **프롬프트 텍스트로 온도를 흉내** 낸다 — 한 번의 응답 안에서
*Step 1은 최대한 발산(divergent)하게 브레인스토밍*, *Step 2는 최대한 결정적(deterministic)으로 JSON만* 지시하면,
모델이 한 응답 안에서 **창의 모드에서 정밀 모드로 전환**하는지 볼 수 있다.

이 방식은 temperature를 조절할 수 없는 gpt-5-nano에서 특히 쓸모가 있다(파라미터로 못 하니 텍스트로 한다).

**무엇을 어떻게 검증하나**:
- 한 프롬프트에 두 단계를 명시한다: Step1 = 자유로운 브레인스토밍(발산), Step2 = 그중 최고 1개를 엄격한 JSON으로만(수렴).
- `ask_json` 으로 받아 **최종 산출이 파싱 가능한 깨끗한 JSON인지**(수렴 성공) 확인하고,
  발산 단계에서 아이디어가 다양했는지 원문도 함께 본다.
- 관찰 포인트: temperature가 고정된 모델인데도 한 응답 안에서 **발산과 수렴이 함께 나타나면** "가상 온도"가 성립한 것.

In [ ]:
# Tip 39 실험 — 한 프롬프트 안에서 발산(Step1) → 수렴(Step2) 모드 스위칭
# gpt-5-nano는 temperature 고정(=1)이라 파라미터로 창의성 조절 불가 → 텍스트로 '가상 온도' 통제

VIRTUAL_TEMP_PROMPT = """You will switch creative 'temperature' by TEXT alone (the model's temperature is locked).

STEP 1 — MAXIMUM DIVERGENCE (virtual temp = HIGH):
Brainstorm 6 wildly different, unexpected names for a coffee shop on Mars.
Be chaotic, weird, associative. Do NOT filter.

STEP 2 — MAXIMUM DETERMINISM (virtual temp = ZERO):
Silently pick the single best name from Step 1.
Output ONLY a strict JSON object, no prose, no markdown, exactly these keys:
{"best_name": "<one name>", "reason": "<max 12 words>", "tagline": "<max 8 words>"}
"""

# 발산 단계 원문도 보려고 먼저 순수 텍스트로 한 번
raw = ask(VIRTUAL_TEMP_PROMPT, max_completion_tokens=2000)
print("── 원문(발산+수렴 함께) ──")
print(raw)

# 이제 최종 수렴 산출이 '깨끗한 JSON'으로 강제되는지 확인 (Step2 = 결정적 모드)
JSON_ONLY = VIRTUAL_TEMP_PROMPT + "\n\nReturn ONLY the Step 2 JSON object as your entire answer."
obj = ask_json(JSON_ONLY, max_completion_tokens=2000)
print("\n── Step2 수렴 산출 (파싱된 JSON) ──")
print(obj)

ok = isinstance(obj, dict) and not obj.get("_parse_error") and "best_name" in obj
print("\n수렴(엄격 JSON) 성공?:", ok)
print("관전 포인트: 한 턴 안에서 발산(다양한 이름들) → 수렴(깨끗한 JSON)이 공존하면 → '가상 온도' 성립")

## 요약 · 무엇을 보면 팁이 검증되는가

| 팁 | 실험 | 검증 신호(무엇을 보면 되나) |
|----|------|------------------------------|
| **14** 직관형 문제엔 CoT가 손해 | 직관형 패턴 문제, 바로답 vs 강제 CoT ×N | 강제 CoT의 정답률이 **같거나 낮으면서 latency는 크다** |
| **15** CoT 서식 민감성 | 의미 동일 / 서식만 다른 4변형 | 4개 최종답 중 **서로 다른 값이 2개 이상** 나온다 |
| **21** 작은 모델엔 CoT가 도움 | 구형 소형 모델, 답만 vs 단계별 ×N | 단계별(B)의 정답률이 **A보다 뚜렷이 높다** |
| **30** 무관한 개념 융합 | 평범 요청 vs 무관한 두 개념 융합 | 융합(B)이 더 이질적·구체적, **두 영역 키워드 동시 적중** |
| **39** 가상 온도 | 한 응답에 발산 Step1 → 수렴 Step2 | temperature 고정인데도 **발산과 깨끗한 JSON 수렴이 공존** |

**핵심 정리**:
- CoT는 어디에나 좋은 게 아니다 — **직관형 문제(Tip14)엔 손해**, **다단계 문제·작은 모델(Tip21)엔 도움**. 문제 성격과 모델 크기에 따라 갈린다.
- CoT는 **표면 형식에 예민**(Tip15)하다 → 프롬프트 서식을 고정하고, 중요한 파이프라인은 형식까지 버전 관리하라.
- `gpt-5-nano`는 temperature가 고정이라 창의성 조절은 **프롬프트 텍스트가 유일한 수단**(Tip30·39)이다.
  발산은 "무관한 개념을 융합하고 걸러내지 마라"로, 수렴은 "오직 JSON만"으로 텍스트에서 온도를 만든다.

> 재현성 메모: `gpt-5-nano`는 temperature=1 고정이라 실행할 때마다 결과가 달라진다.
> 정답률 셀은 N을 키우면 더 안정적이지만 크레딧을 더 쓴다. 숫자 하나가 아니라 **경향**을 보라.